# SpkIOSzero — DRAGON BALL: Sparking! ZERO (corrigido)

**Versão com célula 4 autossuficiente:** ela inicializa o Drive, importa os scripts da branch `SpkIOSzero`, define `HANDOFF_READY` antes de utilizá-lo e recupera ou prepara o handoff. Não depende de variáveis criadas nas células anteriores.

Origem exclusiva: `Drive/<pasta configurada>/spk` para base, update e 10 DLCs .nsp; `prod.keys` pode estar na raiz configurada ou em `spk`.


In [ ]:
import os, sys, argparse, json
from pathlib import Path
from getpass import getpass
from google.colab import auth
import google.auth
from googleapiclient.discovery import build

DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
ROOT = Path("/content/spk-ios-zero")

auth.authenticate_user()
DRIVE_CREDENTIALS, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/drive"]
)
drive_api_client = build(
    "drive", "v3", credentials=DRIVE_CREDENTIALS, cache_discovery=False
)
folder = drive_api_client.files().get(
    fileId=DRIVE_FOLDER_ID, fields="id,name,mimeType"
).execute()
assert folder.get("mimeType") == "application/vnd.google-apps.folder", folder
spk_folder = next(
    (x for x in drive_api_client.files().list(
        q=f"'{DRIVE_FOLDER_ID}' in parents and trashed = false",
        pageSize=100, fields="files(id,name,mimeType)"
    ).execute().get("files", [])
     if x.get("mimeType") == "application/vnd.google-apps.folder"
     and x.get("name", "").lower() == "spk"),
    None,
)
assert spk_folder, "A pasta compartilhada precisa conter a subpasta spk."
root_names = {x.get("name", "").lower() for x in drive_api_client.files().list(
    q=f"'{DRIVE_FOLDER_ID}' in parents and trashed = false",
    pageSize=100, fields="files(id,name,mimeType)"
).execute().get("files", []) if x.get("mimeType") != "application/vnd.google-apps.folder"}
spk_names = {x.get("name", "").lower() for x in drive_api_client.files().list(
    q=f"'{spk_folder['id']}' in parents and trashed = false",
    pageSize=100, fields="files(id,name,mimeType)"
).execute().get("files", []) if x.get("mimeType") != "application/vnd.google-apps.folder"}
assert "prod.keys" in root_names or "prod.keys" in spk_names, "prod.keys precisa estar na raiz configurada ou dentro de spk."
print("Google Drive autenticado:", folder.get("name"))
print("Pasta exclusiva do Sparking! ZERO:", spk_folder.get("name"))
print("Autenticação concluída; nenhum jogo foi baixado nesta célula.")


## 1. Obter o pipeline SpkIOSzero

Clona somente a branch `SpkIOSzero`. O checkout pequeno dos scripts não é o download dos arquivos do jogo.

In [ ]:
%cd /content
!rm -rf /content/SpkIOSzero-repo
!git clone --depth 1 --branch SpkIOSzero https://github.com/joao9vegeto-stack/KintTany.git /content/SpkIOSzero-repo
!git -C /content/SpkIOSzero-repo rev-parse --short HEAD

## 2. Recuperar uma preparação SPK já concluída (sem refazer)

Execute esta célula primeiro ao retomar uma sessão interrompida ou quando a preparação já tiver sido concluída. Ela procura exclusivamente pastas `SpkIOSzero_*` dentro da pasta compartilhada, lê o `handoff.json` e recupera o estado SPK.

**Não baixa NSPs, não extrai conteúdo, não executa AOT rewrite nem shader split.** Pastas `Link2iOS_BOTW_*`, `Kleo2iOS_GameData_*` e arquivos de outros jogos são ignorados.

In [ ]:
%cd /content/SpkIOSzero-repo
import importlib, hashlib, subprocess
sys.path.insert(0, "/content/SpkIOSzero-repo/colab")
# Remove módulos em cache para impedir execução de código de outro checkout.
for module_name in ("source_spk", "prepare_spk", "run_spk"):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
repo_head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd="/content/SpkIOSzero-repo", text=True
).strip()
import source_spk
import prepare_spk
import run_spk
expected_source = Path("/content/SpkIOSzero-repo/colab/source_spk.py").resolve()
loaded_source = Path(source_spk.__file__).resolve()
assert loaded_source == expected_source, f"source_spk carregado de caminho inesperado: {loaded_source}"
source_text = loaded_source.read_text(encoding="utf-8")
assert "update.nsp" in source_text and "dlc" in source_text.lower() and "validate_nsp" in source_text, (
    "O checkout não contém o suporte de origem NSP esperado para o Sparking! ZERO. "
    f"HEAD={repo_head}; arquivo={loaded_source}"
)
assert "len(dlc_items) != 10" in source_text, "A validação de exatamente 10 DLC NSPs está ausente."
print("[SpkIOSzero] Commit do código:", repo_head)
print("[SpkIOSzero] Módulo de origem:", loaded_source)
print("[SpkIOSzero] SHA-256 source_spk.py:", hashlib.sha256(loaded_source.read_bytes()).hexdigest())
RECOVERED_HANDOFF = run_spk.recover(
    argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID),
    api=drive_api_client,
)
HANDOFF_READY = RECOVERED_HANDOFF is not None

## 3. Primeira preparação Linux (executar somente se não houver recuperação)

Esta é a etapa demorada. O Colab baixa o arquivo 7z do jogo base pelo link assinado e extrai o único NSP. Update, exatamente 10 DLCs NSP e `prod.keys` são lidos exclusivamente de `Drive/<pasta configurada>/spk`.

O NXNative lê metadados, extrai o jogo, executa o rewrite AArch64 e prepara RomFS/AOC. O decode Unreal/Oodle, shader split, compilação Metal, framework e Shared Shell ficam para o runner macOS da Route C. O Colab salva o workspace e publica um manifest de recuperação no Drive.

**Não execute esta célula se a etapa 2 recuperou uma preparação válida.**

In [ ]:
# CÉLULA AUTOSSUFICIENTE: não depende de HANDOFF_READY nem das células anteriores.
# Sempre inicializa o estado antes de qualquer leitura e recupera o handoff do Drive.
import os, sys, json, argparse, importlib, subprocess, re
from pathlib import Path
from getpass import getpass

HANDOFF_READY = False
HANDOFF = None
RECOVERED_HANDOFF = None
HANDOFF_PATH = Path("/content/spk-ios-zero-handoff.json")
from google.colab import auth
import google.auth
from googleapiclient.discovery import build

DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
ROOT = Path("/content/spk-ios-zero")

# Autenticação e cliente Drive independentes das células anteriores.
try:
    DRIVE_CREDENTIALS, _ = google.auth.default(
        scopes=["https://www.googleapis.com/auth/drive"]
    )
except Exception:
    auth.authenticate_user()
    DRIVE_CREDENTIALS, _ = google.auth.default(
        scopes=["https://www.googleapis.com/auth/drive"]
    )
drive_api_client = build(
    "drive", "v3", credentials=DRIVE_CREDENTIALS, cache_discovery=False
)

# Obtém os scripts exatamente da branch SpkIOSzero; não importa módulos antigos em cache.
REPO_DIR = Path("/content/SpkIOSzero-repo")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "SpkIOSzero",
        "https://github.com/joao9vegeto-stack/KintTany.git", str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "SpkIOSzero"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "SpkIOSzero"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/SpkIOSzero"], check=True)

sys.path.insert(0, str(REPO_DIR / "colab"))
for module_name in ("source_spk", "prepare_spk", "run_spk"):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
import source_spk
import prepare_spk
import run_spk

repo_head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
assert Path(source_spk.__file__).resolve() == (REPO_DIR / "colab/source_spk.py").resolve(), (
    "O módulo source_spk foi carregado de um caminho inesperado."
)
print("[SpkIOSzero] Branch/commit dos scripts:", repo_head)
print("[SpkIOSzero] DRIVE_FOLDER_ID:", DRIVE_FOLDER_ID)

# Primeiro caminho: recuperar uma preparação já concluída, sem repetir downloads ou rewrite.
RECOVERED_HANDOFF = run_spk.recover(
    argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID),
    api=drive_api_client,
)
if RECOVERED_HANDOFF is not None:
    HANDOFF = RECOVERED_HANDOFF
    HANDOFF_PATH.write_text(json.dumps(HANDOFF, indent=2) + "\n", encoding="utf-8")
    HANDOFF_READY = True
    print("[SpkIOSzero] Handoff anterior recuperado. Download, extração e rewrite serão pulados.")

# Segundo caminho: preparar somente quando não existe handoff recuperável.
if not HANDOFF_READY:
    root_items = run_spk.list_children(drive_api_client, DRIVE_FOLDER_ID)
    folder_mime = "application/vnd.google-apps.folder"
    spk_folder = next(
        (item for item in root_items
         if item.get("mimeType") == folder_mime
         and item.get("name", "").lower() == "spk"),
        None,
    )
    assert spk_folder is not None, (
        "A pasta configurada do Drive precisa conter a subpasta spk."
    )
    spk_items = run_spk.list_children(drive_api_client, spk_folder["id"])
    package_items = [
        item for item in spk_items
        if item.get("mimeType") != folder_mime
        and Path(item.get("name", "")).suffix.lower() in (".nsp", ".xci")
    ]
    update_items = [
        item for item in package_items
        if item.get("name", "").lower() == "update.nsp"
    ]
    assert len(update_items) == 1, (
        f"Esperado exatamente um update.nsp em spk; encontrados {len(update_items)}."
    )
    dlc_items = [
        item for item in package_items
        if item["id"] != update_items[0]["id"]
        and re.fullmatch(r"dlc[^/]*\.nsp", item.get("name", ""), re.I)
    ]
    assert len(dlc_items) == 10, (
        f"Esperados exatamente 10 DLCs .nsp em spk; encontrados {len(dlc_items)}."
    )
    excluded_ids = {update_items[0]["id"], *(item["id"] for item in dlc_items)}
    base_items = [item for item in package_items if item["id"] not in excluded_ids]
    assert len(base_items) <= 1, (
        "Drive/spk pode conter no máximo um arquivo base, .nsp ou .xci; "
        f"foram encontrados {len(base_items)}."
    )

    root_keys = [
        item for item in root_items
        if item.get("mimeType") != folder_mime
        and item.get("name", "").lower() == "prod.keys"
    ]
    spk_keys = [
        item for item in spk_items
        if item.get("mimeType") != folder_mime
        and item.get("name", "").lower() == "prod.keys"
    ]
    # Aceita a cópia em ambas as pastas; prioriza Drive/spk para manter o isolamento do jogo.
    assert len(root_keys) <= 1 and len(spk_keys) <= 1 and (root_keys or spk_keys), (
        "prod.keys não encontrado: coloque-o na raiz configurada ou em Drive/spk. "
        f"Encontrados {len(root_keys)} na raiz e {len(spk_keys)} em spk."
    )
    keys_item = (spk_keys or root_keys)[0]

    cached_archive = (
        ROOT / "7z-temporary" / "sparking-zero-base.7z"
    ).is_file()
    if not base_items and not cached_archive:
        signed_url = getpass(
            "Cole o link assinado atualizado do 7z com o NSP ou XCI base do Sparking! ZERO: "
        ).strip()
        assert signed_url.startswith(("https://", "http://")), (
            "É necessária uma URL HTTPS válida para o 7z do jogo base."
        )
        os.environ["SPK_BASE_ARCHIVE_URL"] = signed_url
        del signed_url
    else:
        os.environ.pop("SPK_BASE_ARCHIVE_URL", None)
        if base_items:
            print("[SpkIOSzero] Base NSP/XCI já existe em Drive/spk; não será baixado o 7z.")
        else:
            print("[SpkIOSzero] 7z já existe nesta sessão; será reutilizado.")

    print("[SpkIOSzero] Iniciando preparação de 1 base NSP/XCI + 1 update NSP + 10 DLC NSPs...")
    run_spk.prepare(
        argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID),
        api=drive_api_client,
    )
    assert HANDOFF_PATH.is_file(), (
        "A preparação terminou, mas /content/spk-ios-zero-handoff.json não foi criado."
    )
    HANDOFF = json.loads(HANDOFF_PATH.read_text(encoding="utf-8"))

# Estado final é definido aqui em todos os caminhos, sem depender de outra célula.
assert HANDOFF is not None, "Nenhum handoff SPK foi recuperado ou criado."
assert HANDOFF.get("project") == "SpkIOSzero"
assert HANDOFF.get("branch") == "SpkIOSzero"
assert HANDOFF.get("title_id") == "010035F022078000"
assert HANDOFF.get("source", {}).get("package_count") == 12
assert HANDOFF.get("source", {}).get("dlc_nsp_count") == 10
assert HANDOFF.get("source", {}).get("base_type") in ("NSP", "XCI")
HANDOFF_READY = True
print("[SpkIOSzero] HANDOFF_READY =", HANDOFF_READY)
print("[SpkIOSzero] Request:", HANDOFF.get("request_id"))
print("[SpkIOSzero] Preparação SPK concluída/verificada.")


## 4. Verificação do resultado e checkpoint de recuperação

O manifest persistido confirma Title ID, versão, hash do workspace, RomFS e imagens AOC extraídas. Também valida a origem: 1 NSP base + 1 update NSP + exatamente 10 DLCs NSP. Se o runtime do Colab reiniciar, basta repetir autenticação + clone + recuperação; não execute novamente a etapa longa.

In [ ]:
HANDOFF_PATH = Path("/content/spk-ios-zero-handoff.json")
assert HANDOFF_READY and HANDOFF_PATH.is_file(), "Nenhum handoff SPK recuperado/gerado."
HANDOFF = json.loads(HANDOFF_PATH.read_text(encoding="utf-8"))
assert HANDOFF["title_id"] == "010035F022078000"
assert HANDOFF["branch"] == "SpkIOSzero"
assert HANDOFF["project"] == "SpkIOSzero"
assert HANDOFF["source"]["package_count"] == 12
assert HANDOFF["source"]["update_nsp_count"] == 1
assert HANDOFF["source"]["dlc_nsp_count"] == 10
assert HANDOFF["prepared"]["input_package_count"] == 12
assert HANDOFF["prepared"]["input_nsp_count"] == 12
assert HANDOFF["prepared"]["input_dlc_nsp_count"] == 10
assert HANDOFF["prepared"]["main_build_id"] == "68edafd68c7bc1a1af50b82aac838018"
assert HANDOFF["prepared"]["main_sha256"] == "a0f8f0644c92384ddb1baac1c21aecbeeace5bc99ad40895e216add61da78efe"
assert {x["name"] for x in HANDOFF["game_data"]["aoc"]} == {x["name"] for x in HANDOFF["prepared"]["aoc_images"]}
print("Sparking! ZERO — preparação verificada.")
print("Request:", HANDOFF["request_id"])
print("Drive:", HANDOFF.get("drive_output_folder_url"))
print("Workspace:", f'{HANDOFF["work_archive"]["size"]:,} bytes')
print("RomFS:", f'{HANDOFF["game_data"]["romfs"]["size"]:,} bytes')
print("DLC NSPs de origem:", HANDOFF["source"]["dlc_nsp_count"])
print("AOC images extraídas:", len(HANDOFF["game_data"]["aoc"]))
print("Pronto para o handoff do GitHub Actions.")

## 5. Última célula: enviar handoff e disparar somente SpkIOSzero

Use o Fine-grained PAT com `Contents: Read and write` restrito ao repositório. Esta célula envia apenas `handoff.json` para a branch `SpkIOSzero`; o workflow final baixa e valida os artefatos preparados no Drive. Não altera `main`, `Link2iOS` nem `Kleo2iOS`.

In [ ]:
%cd /content/SpkIOSzero-repo
!python colab/run_spk.py dispatch --handoff /content/spk-ios-zero-handoff.json